# Advanced: Fed-Batch CHO Bioreactor Model — Simulation and Parameter Exploration
In this notebook, we describe and simulate the kinetic model of a fed-batch Chinese Hamster Ovary (CHO) cell bioreactor producing a monoclonal antibody. We explore how the key model parameters — growth kinetics, by-product inhibition, feed policy, and yield coefficients — influence the bioreactor trajectory: biomass accumulation, substrate consumption, by-product accumulation, and final antibody titer.

> __Learning Objectives:__
>
> By the end of this notebook, you should be able to:
>
> * __Describe the fed-batch CHO bioreactor model:__ Identify the seven state variables, write the mass balance ODEs, and explain how the Monod growth law with by-product inhibition governs cell behavior.
> * __Simulate the CHO model under a glucose-triggered feed policy:__ Use the `simulate_fedbatch(...)` function to integrate the ODE system and visualize the full state trajectory.
> * __Explore parameter sensitivity:__ Vary kinetic parameters, yield coefficients, and feed policy settings and observe how each change affects biomass, antibody titer, and by-product accumulation.

Let's get started!
___

## Setup and Prerequisites
We set up the computational environment by including the `Include.jl` file.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, and includes local source files in `src/`.

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

The CHO bioreactor model is implemented in the `src/` directory. The key functions used in this notebook are:

| Function | Source | Description |
| --- | --- | --- |
| `build_default_parameters(...)` | `src/Parameters.jl` | Construct a `MyFedBatchCHOParameters` instance with default kinetic values |
| `growth_rate(...)` | `src/Kinetics.jl` | Monod growth rate with dual substrate limitation and by-product inhibition |
| `simulate_fedbatch(...)` | `src/Simulation.jl` | Integrate the CHO ODE system with glucose-triggered square wave feeding |

___

## Section 1: The Fed-Batch CHO Bioreactor Model
A fed-batch CHO bioreactor operates by adding a concentrated nutrient feed to the reactor at intervals, rather than draining and refilling it. The reactor volume grows as feed is added, concentrations of nutrients are replenished, and the cells grow, consume substrates, secrete by-products, and produce antibody.

### 1.1 State Variables
The model tracks seven state variables:

| Index | Symbol | Units | Description |
| :---: | :---: | :---: | --- |
| 1 | $V$ | L | Reactor volume |
| 2 | $X$ | gDW/L | Biomass (dry cell weight) concentration |
| 3 | $S_{glc}$ | mM | Glucose concentration |
| 4 | $S_{gln}$ | mM | Glutamine concentration |
| 5 | $P$ | mg/L | Antibody (product) concentration |
| 6 | $Lac$ | mM | Lactate concentration |
| 7 | $Amm$ | mM | Ammonia concentration |

### 1.2 Mass Balance Equations
The governing ODEs are:

> __Fed-batch CHO mass balances__
>
> Let $F$ (L/h) be the volumetric feed rate and $D = F/V$ (1/h) be the dilution rate. The state equations are:
> $$
\begin{align*}
\frac{dV}{dt} &= F \\\\
\frac{dX}{dt} &= (\mu - k_d - D)\,X \\\\
\frac{dS_{glc}}{dt} &= D(S_{glc,f} - S_{glc}) - q_{glc}\,X \\\\
\frac{dS_{gln}}{dt} &= D(S_{gln,f} - S_{gln}) - q_{gln}\,X \\\\
\frac{dP}{dt} &= q_P\,X - D\,P \\\\
\frac{dLac}{dt} &= q_{lac}\,X - D\,Lac \\\\
\frac{dAmm}{dt} &= q_{amm}\,X - D\,Amm
\end{align*}
> $$
> where $\mu$ (1/h) is the specific growth rate, $k_d$ (1/h) is the death rate, $q_P$ (mg/gDW/h) is the specific antibody productivity, and $q_{(\cdot)}$ terms are specific uptake or formation rates.

### 1.3 Growth Rate Model
The specific growth rate follows Monod kinetics with dual substrate limitation and by-product inhibition:

> __Monod growth rate with inhibition__
>
> $$
\mu = \mu_{\max}
\underbrace{\left(\frac{S_{glc}}{K_{glc}+S_{glc}}\right)}_{\text{glucose limitation}}
\underbrace{\left(\frac{S_{gln}}{K_{gln}+S_{gln}}\right)}_{\text{glutamine limitation}}
\underbrace{\left(\frac{K_{I,lac}}{K_{I,lac}+Lac}\right)}_{\text{lactate inhibition}}
\underbrace{\left(\frac{K_{I,amm}}{K_{I,amm}+Amm}\right)}_{\text{ammonia inhibition}}
> $$
> The two substrate terms reduce growth when glucose or glutamine is scarce (Monod saturation). The two inhibition terms reduce growth as lactate and ammonia accumulate (competitive inhibition).

### 1.4 Specific Uptake and Formation Rates
Substrate uptake rates follow a yield-based structure:

> __Yield-based uptake and formation rates__
>
> $$
\begin{align*}
q_{glc} &= \frac{\mu}{Y_{X/glc}} + \frac{q_P}{Y_{P/glc}} \\\\
q_{gln} &= \frac{\mu}{Y_{X/gln}} + \frac{q_P}{Y_{P/gln}} \\\\
q_{lac} &= Y_{lac/glc}\,q_{glc} \\\\
q_{amm} &= Y_{amm/gln}\,q_{gln}
\end{align*}
> $$
> The yield coefficients $Y_{(\cdot)}$ link the rates of substrate consumption and by-product formation to growth and product formation.

### 1.5 Glucose-Triggered Feed Policy
The feed operates as a square wave with hysteresis:

> __Glucose-triggered feeding__
>
> * The feed turns **ON** (at rate $F_{max}$) when glucose drops below $S_{glc,min}$.
> * The feed turns **OFF** when glucose rises above $S_{glc,max}$.
>
> This mimics a common industrial control strategy that avoids both substrate starvation and glucose overflow. The two thresholds $S_{glc,min}$ and $S_{glc,max}$ (along with $F_{max}$) are the feed policy parameters.

### 1.6 Default Parameter Values
The table below lists all kinetic parameters, their default values, and their units:

| Parameter | Symbol | Default Value | Units | Description |
| :---: | :---: | :---: | :---: | --- |
| Maximum growth rate | $\mu_{max}$ | 0.035 | 1/h | Monod maximum growth rate |
| Glucose half-saturation | $K_{glc}$ | 1.5 | mM | Glucose Monod constant |
| Glutamine half-saturation | $K_{gln}$ | 0.12 | mM | Glutamine Monod constant |
| Lactate inhibition constant | $K_{I,lac}$ | 47.0 | mM | Lactate inhibition constant |
| Ammonia inhibition constant | $K_{I,amm}$ | 8.0 | mM | Ammonia inhibition constant |
| Death rate | $k_d$ | 0.005 | 1/h | Specific death rate |
| Specific productivity | $q_P$ | 0.015 | mg/gDW/h | Antibody secretion rate |
| Biomass yield on glucose | $Y_{X/glc}$ | 0.070 | gDW/mmol | Cells per glucose consumed |
| Biomass yield on glutamine | $Y_{X/gln}$ | 0.210 | gDW/mmol | Cells per glutamine consumed |
| Lactate yield on glucose | $Y_{lac/glc}$ | 1.2 | mmol/mmol | Lactate per glucose consumed |
| Ammonia yield on glutamine | $Y_{amm/gln}$ | 0.70 | mmol/mmol | Ammonia per glutamine consumed |
| Feed glucose concentration | $S_{glc,f}$ | 500.0 | mM | Glucose in feed reservoir |
| Feed glutamine concentration | $S_{gln,f}$ | 50.0 | mM | Glutamine in feed reservoir |

___

## Section 2: Baseline Simulation
We run a baseline simulation with default parameter values to establish a reference trajectory. The default parameters are mid-range literature values for CHO fed-batch culture.

In [ ]:
# baseline simulation parameters -
# feed policy: moderate feed rate, tight glucose band -
p_baseline = build_default_parameters(;
    F_max   = 0.02,        # maximum feed rate: 0.02 L/h
    Glc_min = 2.0,         # turn feed ON when glucose < 2 mM
    Glc_max = 20.0,        # turn feed OFF when glucose > 20 mM
);

# initial conditions: [V (L), X (gDW/L), S_glc (mM), S_gln (mM), P (mg/L), Lac (mM), Amm (mM)] -
u0_baseline = [0.5, 0.5, 25.0, 4.0, 0.0, 0.0, 0.0];
tspan = (0.0, 240.0);   # 240 hours = 10 days
saveat = 1.0;            # save every 1 hour

# run simulation -
sol_baseline = simulate_fedbatch(p_baseline; u0 = u0_baseline, tspan = tspan, saveat = saveat);
t = sol_baseline.t;
println("Simulation complete: $(length(t)) time points over $(tspan[2]) hours");

Let's plot all seven state variables for the baseline run.

In [ ]:
let
    sol = sol_baseline;
    t = sol.t;

    # extract states -
    V     = [sol.u[i][1] for i in 1:length(t)];
    X     = [sol.u[i][2] for i in 1:length(t)];
    S_glc = [sol.u[i][3] for i in 1:length(t)];
    S_gln = [sol.u[i][4] for i in 1:length(t)];
    P     = [sol.u[i][5] for i in 1:length(t)];
    Lac   = [sol.u[i][6] for i in 1:length(t)];
    Amm   = [sol.u[i][7] for i in 1:length(t)];

    # panel plots -
    p1 = plot(t, V,     ylabel="V (L)",          label=false, lw=2, color=:steelblue,   title="Volume");
    p2 = plot(t, X,     ylabel="X (gDW/L)",      label=false, lw=2, color=:darkorange,  title="Biomass");
    p3 = plot(t, S_glc, ylabel="Glucose (mM)",   label=false, lw=2, color=:green4,      title="Glucose");
    p4 = plot(t, S_gln, ylabel="Glutamine (mM)", label=false, lw=2, color=:purple,      title="Glutamine");
    p5 = plot(t, P,     ylabel="P (mg/L)",       label=false, lw=2, color=:crimson,     title="Antibody");
    p6 = plot(t, Lac,   ylabel="Lactate (mM)",   label=false, lw=2, color=:saddlebrown, title="Lactate");
    p7 = plot(t, Amm,   ylabel="Ammonia (mM)",   label=false, lw=2, color=:slateblue,   title="Ammonia");

    # add xlabel to bottom row panels -
    for p in [p5, p6, p7]
        plot!(p, xlabel="Time (h)")
    end

    display(plot(p1, p2, p3, p4, p5, p6, p7, layout=(3, 3), size=(1000, 700),
        plot_title="Baseline Fed-Batch CHO Simulation"))
end

___
## Section 3: Growth Kinetics Exploration
We explore how the maximum growth rate $\mu_{max}$ affects the bioreactor trajectory. All other parameters are held at baseline values.

In [ ]:
let
    # vary mu_max: low, baseline, high -
    mu_max_values = [0.025, 0.035, 0.050]; # 1/h
    colors = [:royalblue, :black, :firebrick];
    labels = ["μ_max = 0.025 (low)", "μ_max = 0.035 (baseline)", "μ_max = 0.050 (high)"];

    p_biomass = plot(ylabel="X (gDW/L)",    xlabel="Time (h)", title="Effect of μ_max on Biomass",       legend=:topleft);
    p_product = plot(ylabel="P (mg/L)",     xlabel="Time (h)", title="Effect of μ_max on Antibody Titer", legend=:topleft);
    p_glucose = plot(ylabel="Glucose (mM)", xlabel="Time (h)", title="Effect of μ_max on Glucose",       legend=:topright);
    p_lactate = plot(ylabel="Lactate (mM)", xlabel="Time (h)", title="Effect of μ_max on Lactate",       legend=:topleft);

    for (i, mu) in enumerate(mu_max_values)
        p = build_default_parameters(; F_max=0.02, Glc_min=2.0, Glc_max=20.0, mu_max=mu);
        sol = simulate_fedbatch(p; u0=copy(u0_baseline), tspan=tspan, saveat=saveat);
        X_t   = [sol.u[j][2] for j in 1:length(sol.t)];
        P_t   = [sol.u[j][5] for j in 1:length(sol.t)];
        Glc_t = [sol.u[j][3] for j in 1:length(sol.t)];
        Lac_t = [sol.u[j][6] for j in 1:length(sol.t)];
        plot!(p_biomass, sol.t, X_t,   label=labels[i], lw=2, color=colors[i]);
        plot!(p_product, sol.t, P_t,   label=labels[i], lw=2, color=colors[i]);
        plot!(p_glucose, sol.t, Glc_t, label=labels[i], lw=2, color=colors[i]);
        plot!(p_lactate, sol.t, Lac_t, label=labels[i], lw=2, color=colors[i]);
    end

    display(plot(p_biomass, p_product, p_glucose, p_lactate, layout=(2, 2), size=(900, 650)))
end

___
## Section 4: By-Product Inhibition Exploration
Lactate and ammonia accumulate as by-products and inhibit growth. We explore how the inhibition constants $K_{I,lac}$ and $K_{I,amm}$ control the sensitivity of growth to by-product accumulation. A smaller inhibition constant means stronger inhibition at the same by-product concentration.

In [ ]:
let
    # vary K_I_lac (lactate inhibition constant) -
    KI_lac_values = [20.0, 47.0, 80.0]; # mM
    colors = [:firebrick, :black, :royalblue];
    labels_lac = ["K_I,lac = 20 mM (sensitive)", "K_I,lac = 47 mM (baseline)", "K_I,lac = 80 mM (tolerant)"];

    p1 = plot(ylabel="X (gDW/L)",    xlabel="Time (h)", title="Biomass: Lactate Inhibition", legend=:topleft);
    p2 = plot(ylabel="Lactate (mM)", xlabel="Time (h)", title="Lactate: Lactate Inhibition", legend=:topleft);

    for (i, KI) in enumerate(KI_lac_values)
        p = build_default_parameters(; F_max=0.02, Glc_min=2.0, Glc_max=20.0, K_I_lac=KI);
        sol = simulate_fedbatch(p; u0=copy(u0_baseline), tspan=tspan, saveat=saveat);
        X_t   = [sol.u[j][2] for j in 1:length(sol.t)];
        Lac_t = [sol.u[j][6] for j in 1:length(sol.t)];
        plot!(p1, sol.t, X_t,   label=labels_lac[i], lw=2, color=colors[i]);
        plot!(p2, sol.t, Lac_t, label=labels_lac[i], lw=2, color=colors[i]);
    end

    # vary K_I_amm (ammonia inhibition constant) -
    KI_amm_values = [3.0, 8.0, 15.0]; # mM
    labels_amm = ["K_I,amm = 3 mM (sensitive)", "K_I,amm = 8 mM (baseline)", "K_I,amm = 15 mM (tolerant)"];

    p3 = plot(ylabel="X (gDW/L)",    xlabel="Time (h)", title="Biomass: Ammonia Inhibition", legend=:topleft);
    p4 = plot(ylabel="Ammonia (mM)", xlabel="Time (h)", title="Ammonia: Ammonia Inhibition", legend=:topleft);

    for (i, KI) in enumerate(KI_amm_values)
        p = build_default_parameters(; F_max=0.02, Glc_min=2.0, Glc_max=20.0, K_I_amm=KI);
        sol = simulate_fedbatch(p; u0=copy(u0_baseline), tspan=tspan, saveat=saveat);
        X_t   = [sol.u[j][2] for j in 1:length(sol.t)];
        Amm_t = [sol.u[j][7] for j in 1:length(sol.t)];
        plot!(p3, sol.t, X_t,   label=labels_amm[i], lw=2, color=colors[i]);
        plot!(p4, sol.t, Amm_t, label=labels_amm[i], lw=2, color=colors[i]);
    end

    display(plot(p1, p2, p3, p4, layout=(2, 2), size=(900, 650)))
end

___
## Section 5: Feed Policy Exploration
The feed policy — maximum feed rate $F_{max}$, and the glucose thresholds $S_{glc,min}$ and $S_{glc,max}$ — determines when and how much nutrient is added. We explore how each parameter affects the simulation.

In [ ]:
let
    # vary F_max -
    F_max_values = [0.005, 0.02, 0.05]; # L/h
    colors = [:royalblue, :black, :firebrick];
    labels_F = ["F_max = 0.005 L/h (slow)", "F_max = 0.02 L/h (baseline)", "F_max = 0.05 L/h (fast)"];

    p_X = plot(ylabel="X (gDW/L)",    xlabel="Time (h)", title="Biomass: F_max",  legend=:topleft);
    p_P = plot(ylabel="P (mg/L)",     xlabel="Time (h)", title="Antibody: F_max", legend=:topleft);
    p_V = plot(ylabel="V (L)",        xlabel="Time (h)", title="Volume: F_max",   legend=:topleft);
    p_G = plot(ylabel="Glucose (mM)", xlabel="Time (h)", title="Glucose: F_max",  legend=:topright);

    for (i, Fmax) in enumerate(F_max_values)
        p = build_default_parameters(; F_max=Fmax, Glc_min=2.0, Glc_max=20.0);
        sol = simulate_fedbatch(p; u0=copy(u0_baseline), tspan=tspan, saveat=saveat);
        X_t = [sol.u[j][2] for j in 1:length(sol.t)];
        P_t = [sol.u[j][5] for j in 1:length(sol.t)];
        V_t = [sol.u[j][1] for j in 1:length(sol.t)];
        G_t = [sol.u[j][3] for j in 1:length(sol.t)];
        plot!(p_X, sol.t, X_t, label=labels_F[i], lw=2, color=colors[i]);
        plot!(p_P, sol.t, P_t, label=labels_F[i], lw=2, color=colors[i]);
        plot!(p_V, sol.t, V_t, label=labels_F[i], lw=2, color=colors[i]);
        plot!(p_G, sol.t, G_t, label=labels_F[i], lw=2, color=colors[i]);
    end

    display(plot(p_X, p_P, p_V, p_G, layout=(2, 2), size=(900, 650)))
end

In [ ]:
let
    # vary glucose setpoint band (Glc_min, Glc_max) -
    bands = [(1.0, 5.0), (2.0, 20.0), (5.0, 30.0)];
    colors = [:royalblue, :black, :firebrick];
    labels_band = ["Tight: [1, 5] mM", "Baseline: [2, 20] mM", "Wide: [5, 30] mM"];

    p_X = plot(ylabel="X (gDW/L)",    xlabel="Time (h)", title="Biomass: Glucose Band",  legend=:topleft);
    p_P = plot(ylabel="P (mg/L)",     xlabel="Time (h)", title="Antibody: Glucose Band", legend=:topleft);
    p_G = plot(ylabel="Glucose (mM)", xlabel="Time (h)", title="Glucose: Glucose Band",  legend=:topright);
    p_L = plot(ylabel="Lactate (mM)", xlabel="Time (h)", title="Lactate: Glucose Band",  legend=:topleft);

    for (i, (gmin, gmax)) in enumerate(bands)
        p = build_default_parameters(; F_max=0.02, Glc_min=gmin, Glc_max=gmax);
        sol = simulate_fedbatch(p; u0=copy(u0_baseline), tspan=tspan, saveat=saveat);
        X_t = [sol.u[j][2] for j in 1:length(sol.t)];
        P_t = [sol.u[j][5] for j in 1:length(sol.t)];
        G_t = [sol.u[j][3] for j in 1:length(sol.t)];
        L_t = [sol.u[j][6] for j in 1:length(sol.t)];
        plot!(p_X, sol.t, X_t, label=labels_band[i], lw=2, color=colors[i]);
        plot!(p_P, sol.t, P_t, label=labels_band[i], lw=2, color=colors[i]);
        plot!(p_G, sol.t, G_t, label=labels_band[i], lw=2, color=colors[i]);
        plot!(p_L, sol.t, L_t, label=labels_band[i], lw=2, color=colors[i]);
    end

    display(plot(p_X, p_P, p_G, p_L, layout=(2, 2), size=(900, 650)))
end

___
## Section 6: Yield Coefficient Exploration
The yield coefficients $Y_{X/glc}$, $Y_{lac/glc}$, and $Y_{amm/gln}$ link substrate consumption and by-product formation to growth. A higher biomass yield on glucose means more cells per mmol glucose consumed. A higher lactate yield on glucose means more lactate produced per unit glucose consumed.

In [ ]:
let
    colors = [:royalblue, :black, :firebrick];

    # vary Y_lac_glc (lactate yield on glucose) -
    Y_lac_values = [0.7, 1.2, 1.6]; # mmol/mmol
    labels_Ylac = ["Y_lac/glc = 0.7 (low)", "Y_lac/glc = 1.2 (baseline)", "Y_lac/glc = 1.6 (high)"];

    p1 = plot(ylabel="Lactate (mM)", xlabel="Time (h)", title="Lactate: Y_lac/glc", legend=:topleft);
    p2 = plot(ylabel="X (gDW/L)",   xlabel="Time (h)", title="Biomass: Y_lac/glc", legend=:topleft);

    for (i, Ylac) in enumerate(Y_lac_values)
        p = build_default_parameters(; F_max=0.02, Glc_min=2.0, Glc_max=20.0, Y_lac_glc=Ylac);
        sol = simulate_fedbatch(p; u0=copy(u0_baseline), tspan=tspan, saveat=saveat);
        L_t = [sol.u[j][6] for j in 1:length(sol.t)];
        X_t = [sol.u[j][2] for j in 1:length(sol.t)];
        plot!(p1, sol.t, L_t, label=labels_Ylac[i], lw=2, color=colors[i]);
        plot!(p2, sol.t, X_t, label=labels_Ylac[i], lw=2, color=colors[i]);
    end

    # vary Y_X_glc (biomass yield on glucose) -
    Y_X_glc_values = [0.035, 0.070, 0.12]; # gDW/mmol
    labels_YXglc = ["Y_X/glc = 0.035 (low)", "Y_X/glc = 0.070 (baseline)", "Y_X/glc = 0.12 (high)"];

    p3 = plot(ylabel="X (gDW/L)", xlabel="Time (h)", title="Biomass: Y_X/glc",   legend=:topleft);
    p4 = plot(ylabel="P (mg/L)",  xlabel="Time (h)", title="Antibody: Y_X/glc",  legend=:topleft);

    for (i, YXglc) in enumerate(Y_X_glc_values)
        p = build_default_parameters(; F_max=0.02, Glc_min=2.0, Glc_max=20.0, Y_X_glc=YXglc);
        sol = simulate_fedbatch(p; u0=copy(u0_baseline), tspan=tspan, saveat=saveat);
        X_t = [sol.u[j][2] for j in 1:length(sol.t)];
        P_t = [sol.u[j][5] for j in 1:length(sol.t)];
        plot!(p3, sol.t, X_t, label=labels_YXglc[i], lw=2, color=colors[i]);
        plot!(p4, sol.t, P_t, label=labels_YXglc[i], lw=2, color=colors[i]);
    end

    display(plot(p1, p2, p3, p4, layout=(2, 2), size=(900, 650)))
end

___
## Section 7: End-Point Summary Table
We run a grid of conditions and compare the final antibody titer, peak biomass, peak lactate, and peak ammonia across scenarios. This gives a quantitative summary of the parameter sweeps above.

In [ ]:
let
    # define scenarios as named tuples of keyword arguments to build_default_parameters -
    scenarios = [
        (label="Baseline",            F_max=0.02,  Glc_min=2.0, Glc_max=20.0, extra=NamedTuple()),
        (label="Fast feed",           F_max=0.05,  Glc_min=2.0, Glc_max=20.0, extra=NamedTuple()),
        (label="Slow feed",           F_max=0.005, Glc_min=2.0, Glc_max=20.0, extra=NamedTuple()),
        (label="Tight glucose band",  F_max=0.02,  Glc_min=1.0, Glc_max=5.0,  extra=NamedTuple()),
        (label="Wide glucose band",   F_max=0.02,  Glc_min=5.0, Glc_max=30.0, extra=NamedTuple()),
        (label="High μ_max",          F_max=0.02,  Glc_min=2.0, Glc_max=20.0, extra=(mu_max=0.050,)),
        (label="Low μ_max",           F_max=0.02,  Glc_min=2.0, Glc_max=20.0, extra=(mu_max=0.025,)),
        (label="Lac-sensitive cells", F_max=0.02,  Glc_min=2.0, Glc_max=20.0, extra=(K_I_lac=20.0,)),
        (label="Lac-tolerant cells",  F_max=0.02,  Glc_min=2.0, Glc_max=20.0, extra=(K_I_lac=80.0,)),
        (label="High Y_lac/glc",      F_max=0.02,  Glc_min=2.0, Glc_max=20.0, extra=(Y_lac_glc=1.6,)),
        (label="Low Y_lac/glc",       F_max=0.02,  Glc_min=2.0, Glc_max=20.0, extra=(Y_lac_glc=0.7,)),
        (label="High q_P",            F_max=0.02,  Glc_min=2.0, Glc_max=20.0, extra=(q_P=0.030,)),
    ];

    rows = NamedTuple[];
    for s in scenarios
        p = build_default_parameters(; F_max=s.F_max, Glc_min=s.Glc_min, Glc_max=s.Glc_max, s.extra...);
        sol = simulate_fedbatch(p; u0=copy(u0_baseline), tspan=tspan, saveat=saveat);

        P_final  = sol.u[end][5];
        X_peak   = maximum(sol.u[i][2] for i in 1:length(sol.t));
        Lac_peak = maximum(sol.u[i][6] for i in 1:length(sol.t));
        Amm_peak = maximum(sol.u[i][7] for i in 1:length(sol.t));

        push!(rows, (
            Scenario         = s.label,
            Titer_mg_L       = round(P_final,  digits=1),
            Peak_Biomass_gDW = round(X_peak,   digits=2),
            Peak_Lac_mM      = round(Lac_peak, digits=1),
            Peak_Amm_mM      = round(Amm_peak, digits=2),
        ));
    end

    df = DataFrame(rows);
    pretty_table(df;
        header=["Scenario", "Final Titer (mg/L)", "Peak Biomass (gDW/L)", "Peak Lactate (mM)", "Peak Ammonia (mM)"],
        alignment=[:l, :r, :r, :r, :r]
    )
end

___
## Summary
In this notebook, we described and simulated the fed-batch CHO bioreactor model and explored how kinetic parameters, yield coefficients, and feed policy settings affect the bioreactor trajectory.

> __Key Takeaways:__
>
> * **The CHO model couples seven state variables through yield-based kinetics:** Volume, biomass, two substrates, product, and two by-products are linked via the Monod growth law with by-product inhibition and a yield-based stoichiometry. Changing any single parameter propagates through all states.
> * **Feed policy controls both productivity and by-product accumulation:** A faster feed rate supports more biomass and higher titer but also increases lactate and ammonia. The glucose band width determines how tightly glucose is regulated and shapes the feeding pattern.
> * **By-product inhibition constants set a ceiling on achievable biomass:** Low inhibition constants make the culture sensitive to accumulated lactate or ammonia, causing growth arrest earlier in the run. Higher inhibition constants allow the culture to sustain growth at higher by-product concentrations.

This model is the data-generating process used in the [L12c LSTM example](CHEME-5820-L12c-Example-LSTM-CHO-Spring-2026.ipynb), which trains an LSTM to predict the full bioreactor trajectory from the feed policy parameters alone.
___